# Phase 1b — TSLA News Headlines Collection
**Goal:** Collect TSLA-related news headlines from free RSS feeds and save them aligned to trading dates.

**Sources used:**
- Yahoo Finance RSS (TSLA-specific)
- Google News RSS (TSLA keyword search)
- Seeking Alpha RSS

No API key needed for any of these.

## Step 1 — Install and import libraries

In [ ]:
!pip install feedparser pandas requests --quiet

In [ ]:
import feedparser
import pandas as pd
import requests
import time
from datetime import datetime, timedelta
from email.utils import parsedate_to_datetime
import warnings
warnings.filterwarnings('ignore')

print('Libraries ready!')

## Step 2 — Define RSS feed sources

RSS feeds return headlines in XML format. `feedparser` parses them into Python dicts for us — very easy to work with.

In [ ]:
# RSS feed URLs for TSLA news
RSS_FEEDS = {
    'yahoo_finance': 'https://feeds.finance.yahoo.com/rss/2.0/headline?s=TSLA&region=US&lang=en-US',
    'google_news':   'https://news.google.com/rss/search?q=Tesla+TSLA+stock&hl=en-US&gl=US&ceid=US:en',
    'seeking_alpha': 'https://seekingalpha.com/api/sa/combined/TSLA.xml',
}

print('Feed URLs defined.')
print('Note: RSS feeds only return recent headlines (last 30-90 days).')
print('We will handle historical data using a supplementary dataset in Step 5.')

## Step 3 — Fetch headlines from RSS feeds

In [ ]:
def parse_date(entry):
    """Extract and normalise publication date from an RSS entry."""
    # feedparser stores date in 'published' field (sometimes 'updated')
    for field in ['published', 'updated']:
        if hasattr(entry, field):
            try:
                dt = parsedate_to_datetime(getattr(entry, field))
                return dt.strftime('%Y-%m-%d')
            except Exception:
                pass
    return None


def fetch_rss_headlines(feed_name, feed_url):
    """Fetch headlines from a single RSS feed, return list of dicts."""
    print(f'Fetching from {feed_name}...')
    try:
        feed = feedparser.parse(feed_url)
        entries = []
        for entry in feed.entries:
            date = parse_date(entry)
            if date is None:
                continue
            entries.append({
                'date':     date,
                'headline': entry.title.strip(),
                'source':   feed_name,
            })
        print(f'  Got {len(entries)} headlines')
        return entries
    except Exception as e:
        print(f'  Failed: {e}')
        return []


# Fetch from all sources
all_headlines = []
for name, url in RSS_FEEDS.items():
    headlines = fetch_rss_headlines(name, url)
    all_headlines.extend(headlines)
    time.sleep(1)  # Be polite to servers

print(f'\nTotal raw headlines collected: {len(all_headlines)}')

## Step 4 — Convert to DataFrame and clean

In [ ]:
news_df = pd.DataFrame(all_headlines)

if len(news_df) == 0:
    print('WARNING: No headlines collected from RSS. This can happen if feeds are temporarily unavailable.')
    print('Skipping to Step 5 which loads a backup historical dataset.')
else:
    # Sort by date
    news_df = news_df.sort_values('date').reset_index(drop=True)

    # Remove exact duplicates (same headline from multiple sources)
    before = len(news_df)
    news_df.drop_duplicates(subset=['headline'], inplace=True)
    after = len(news_df)
    print(f'Removed {before - after} duplicate headlines')

    # Basic text cleaning
    news_df['headline'] = news_df['headline'].str.strip()

    # Filter to TSLA date range
    news_df = news_df[(news_df['date'] >= '2020-01-01') & (news_df['date'] <= '2024-12-31')]

    print(f'\nClean headlines: {len(news_df)}')
    print(f'Date range: {news_df["date"].min()} to {news_df["date"].max()}')
    print(f'Sources: {news_df["source"].value_counts().to_dict()}')
    news_df.head(10)

## Step 5 — Load historical TSLA news (Kaggle backup dataset)

RSS feeds only give recent headlines. For the 2020–2024 historical range we need a pre-collected dataset.

We use the **'Daily Financial News for 6000+ Stocks'** dataset from Kaggle which has TSLA headlines dating back to 2018.

**Download instructions (one-time):**
1. Go to: https://www.kaggle.com/datasets/miguelaenlle/massive-stock-news-analysis-db-for-nlpbacktests
2. Download `analyst_ratings_processed.csv`
3. Upload it to your Colab session (Files panel on left → Upload)

Then run the cell below.

In [ ]:
import os

KAGGLE_FILE = 'analyst_ratings_processed.csv'

if os.path.exists(KAGGLE_FILE):
    print('Kaggle file found! Loading...')

    kaggle_df = pd.read_csv(KAGGLE_FILE)
    print(f'Total rows in file: {len(kaggle_df)}')
    print(f'Columns: {list(kaggle_df.columns)}')
    print(kaggle_df.head(3))

    # Filter for TSLA only
    # Column names may vary — adjust 'ticker' and 'title' to match your file
    ticker_col   = 'stock'    # change if needed
    headline_col = 'title'    # change if needed
    date_col     = 'date'     # change if needed

    tsla_kaggle = kaggle_df[kaggle_df[ticker_col] == 'TSLA'].copy()
    tsla_kaggle = tsla_kaggle[[date_col, headline_col]].rename(
        columns={date_col: 'date', headline_col: 'headline'}
    )
    tsla_kaggle['source'] = 'kaggle_historical'

    # Normalise date format to YYYY-MM-DD
    tsla_kaggle['date'] = pd.to_datetime(tsla_kaggle['date']).dt.strftime('%Y-%m-%d')

    # Filter to our date range
    tsla_kaggle = tsla_kaggle[
        (tsla_kaggle['date'] >= '2020-01-01') &
        (tsla_kaggle['date'] <= '2024-12-31')
    ]

    print(f'\nTSLA headlines from Kaggle dataset: {len(tsla_kaggle)}')
    print(f'Date range: {tsla_kaggle["date"].min()} to {tsla_kaggle["date"].max()}')
    tsla_kaggle.head(5)

else:
    print(f'File "{KAGGLE_FILE}" not found.')
    print('Please download and upload it following the instructions above.')
    print('\nFor now, we will create a small demo dataset so you can test the pipeline.')

    # Demo fallback — synthetic headlines for pipeline testing only
    # Replace this with real data before training
    demo_dates = pd.date_range('2023-01-01', periods=30, freq='B').strftime('%Y-%m-%d')
    demo_headlines = [
        'Tesla beats Q4 earnings expectations with record deliveries',
        'Elon Musk sells Tesla shares worth $3.6 billion',
        'Tesla recalls vehicles over autopilot concerns',
        'Tesla opens new Gigafactory in Mexico',
        'Tesla stock falls amid broader market selloff',
    ] * 6
    tsla_kaggle = pd.DataFrame({
        'date':     demo_dates,
        'headline': demo_headlines[:30],
        'source':   'demo'
    })
    print(f'\nDemo dataset created with {len(tsla_kaggle)} rows for pipeline testing.')

## Step 6 — Merge RSS and historical headlines

In [ ]:
# Combine RSS (recent) + Kaggle (historical)
frames = [tsla_kaggle]
if len(news_df) > 0:
    frames.append(news_df)

combined = pd.concat(frames, ignore_index=True)

# Final deduplication and clean
combined.drop_duplicates(subset=['headline'], inplace=True)
combined.dropna(subset=['headline', 'date'], inplace=True)
combined = combined.sort_values('date').reset_index(drop=True)

print(f'Combined news dataset: {len(combined)} headlines')
print(f'Date range: {combined["date"].min()} to {combined["date"].max()}')
print(f'Headlines per source:\n{combined["source"].value_counts()}')
combined.head(10)

## Step 7 — Check coverage vs trading days

In [ ]:
# Load price data to check alignment
price_df = pd.read_csv('tsla_price_data.csv')
trading_days = set(price_df['Date'])
news_days    = set(combined['date'])

covered     = trading_days & news_days
not_covered = trading_days - news_days

print(f'Total trading days in price data:  {len(trading_days)}')
print(f'Trading days WITH news coverage:   {len(covered)} ({100*len(covered)/len(trading_days):.1f}%)')
print(f'Trading days WITHOUT news:         {len(not_covered)}')
print()

# Headlines per trading day stats
daily_counts = combined[combined['date'].isin(trading_days)].groupby('date').size()
print(f'Headlines per day — mean: {daily_counts.mean():.1f}, max: {daily_counts.max()}, min: {daily_counts.min()}')

# Plot coverage
import matplotlib.pyplot as plt
daily_counts_full = combined.groupby('date').size().reset_index(name='count')
daily_counts_full['date'] = pd.to_datetime(daily_counts_full['date'])

plt.figure(figsize=(14, 4))
plt.bar(daily_counts_full['date'], daily_counts_full['count'], width=1, color='steelblue', alpha=0.7)
plt.xlabel('Date')
plt.ylabel('Number of headlines')
plt.title('TSLA news headlines per day (2020–2024)')
plt.tight_layout()
plt.savefig('tsla_news_coverage.png', dpi=150)
plt.show()

## Step 8 — Save final news dataset

In [ ]:
combined.to_csv('tsla_news_raw.csv', index=False)

print('Saved: tsla_news_raw.csv')
print(f'Shape: {combined.shape}')
print(f'Columns: {list(combined.columns)}')

## Summary

You now have two clean CSV files:

| File | Contents |
|---|---|
| `tsla_price_data.csv` | ~980 rows, OHLCV + 10 technical indicators |
| `tsla_news_raw.csv` | Headlines with date + source columns |

**What's next — Phase 2:** We'll load `tsla_news_raw.csv` and run every headline through FinBERT to get sentiment scores (positive / negative / neutral), then aggregate them to one score per trading day.

That's where HuggingFace comes in for the first time!